# Chapter 5 (Remaining Parts): Interactive Visualization, Dashboards & Storytelling

- **5.4 Plotly**: interactive plots (hover, zoom, tooltips)
- **5.5 Dashboard thinking**: KPIs, layout, filtering (in a notebook)
- **5.6 Storytelling with data**: highlight, annotate, guide attention
- **Mini Project**: build a tiny “sales dashboard” + narrative story

> Keep datasets small and relatable so concepts are clear.


In [1]:
# pip install plotly
# !pip install statsmodels

## Setup

In [2]:
import numpy as np
import pandas as pd

# Plotly for interactivity
import plotly.express as px # Quick interactive plots (line, bar, scatter) with minimal code
import plotly.graph_objects as go # Fine-grained control over Plotly charts (custom annotations, shapes, advanced layouts)

np.random.seed(42)


---  
## 5.4 Plotly: Interactive Visualization

### Why Plotly?
- Hover to see exact values
- Zoom & pan to inspect regions
- Easy legends, tooltips, and interactive exploration


### 5.4.1 Interactive line chart (trend + hover)

In [3]:
df = pd.DataFrame({
    "day": np.arange(1, 21),
    "sales": [12,13,14,15,13,16,18,17,20,22,21,23,22,24,26,25,27,29,28,30],
    "region": ["East"]*10 + ["West"]*10
})

fig = px.line(df, x="day", y="sales", color="region", markers=True,
              title="Interactive Sales Trend (hover + zoom)")
fig.show()


In [4]:
# Method 1: plotly.express — fast and easy (one line!)
fig1 = px.line(df, x="day", y="sales", title="plotly.express: Simple & Quick")
fig1.show()

# Method 2: graph_objects — more control (more code, but full customization)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df["day"], 
    y=df["sales"], 
    mode="lines+markers",
    line=dict(color="red", width=3, dash="dash"),
    marker=dict(size=10, symbol="diamond", color="darkred")
))
fig2.update_layout(
    title="graph_objects: Full Control",
    xaxis_title="Day",
    yaxis_title="Sales"
)
fig2.show()

**Try:**  
1) Zoom into days 8–15.  
2) Hover on points and read values.  
3) Turn regions on/off using the legend.


### 5.4.2 Interactive scatter plot (relationship + trendline)

In [5]:
df_scatter = pd.DataFrame({
    "ads_budget": np.random.randint(10, 101, 80),
    "sales": 50 + np.random.randint(10, 101, 80) + np.random.normal(0, 10, 80)
})
df_scatter["sales"] = df_scatter["sales"] + 0.6 * df_scatter["ads_budget"]

fig = px.scatter(df_scatter, x="ads_budget", y="sales",
                 trendline="ols",
                 title="Ads Budget vs Sales (Interactive Scatter + Trendline)",
                 labels={"ads_budget":"Ads Budget (k$)", "sales":"Sales (units)"})
fig.show()


## OLS (Ordinary Least Squares) Regression

### What is OLS?
OLS is the most common method for fitting a **linear regression line** to data. It finds the line that **minimizes the sum of squared errors** (distances between actual points and the line).

### The Math
The goal is to find the best $m$ (slope) and $b$ (intercept) for:

$$y = mx + b$$

OLS minimizes:

$$\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Where:
- $y_i$ = actual value
- $\hat{y}_i$ = predicted value from the line


### Why "Least Squares"?
| Method | What it minimizes |
|--------|-------------------|
| **Least Squares** | Sum of squared errors $(y - \hat{y})^2$ |

Squaring the errors:
1. Makes all errors positive (no cancellation)
2. Penalizes large errors more than small ones
3. Has nice mathematical properties (differentiable)

### In Plotly
```python
px.scatter(df, x="ads_budget", y="sales", trendline="ols")

Key Output: R² (R-squared)
R² = 1.0: Perfect fit (all points on the line)
R² = 0.0: No linear relationship
R² = 0.7: 70% of variance in Y is explained by X


**Discuss:**  
- Does higher budget *always* mean higher sales?  
- Why do we still see scatter even with a trendline?


### 5.4.3 Interactive bar chart (category comparison + hover)

In [6]:
df_bar = pd.DataFrame({
    "product": ["A","B","C","D"],
    "revenue": [210, 180, 260, 200]
})

fig = px.bar(df_bar, x="product", y="revenue",
             title="Revenue by Product (Interactive Bar)",
             labels={"revenue":"Revenue (k$)"})
fig.show()


---  
## 5.5 Dashboard Thinking (in a Notebook)

A dashboard should answer:
1) **What is happening?** (KPIs)  
2) **Where/when is it happening?** (breakdowns)  
3) **Why is it happening?** (relationships / drill-down)

### Common dashboard parts
- KPI cards (Total Sales, Avg Daily Sales, Best Product)
- Trend line (time)
- Breakdown bar chart (product/region)
- Filters (date range, region, product)


### 5.5.1 Create a tiny 'sales' dataset for a dashboard

In [7]:
dates = pd.date_range("2025-01-01", periods=90, freq="D")
regions = ["East", "West", "North"]
products = ["A", "B", "C"]

rows = []
for d in dates:
    for r in regions:
        for p in products:
            base = {"A": 20, "B": 16, "C": 24}[p]
            reg_boost = {"East": 2, "West": 0, "North": 1}[r]
            noise = np.random.normal(0, 3)
            sales = max(0, base + reg_boost + noise)
            rows.append([d, r, p, sales])

dash_df = pd.DataFrame(rows, columns=["date", "region", "product", "sales"])
dash_df.head()


,date,region,product,sales
0,2025-01-01,East,A,24.890128
1,2025-01-01,East,B,19.238343
2,2025-01-01,East,C,28.466180
3,2025-01-01,West,A,25.690379
4,2025-01-01,West,B,15.263836


### 5.5.2 KPI calculations (what to show on top of a dashboard)

In [8]:
total_sales = dash_df["sales"].sum()
avg_daily_sales = dash_df.groupby("date")["sales"].sum().mean()
best_product = dash_df.groupby("product")["sales"].sum().idxmax()

print("Total Sales:", round(total_sales, 2))
print("Avg Daily Sales:", round(avg_daily_sales, 2))
print("Best Product:", best_product)


Total Sales: 17119.25
Avg Daily Sales: 190.21
Best Product: C


### 5.5.3 One-page dashboard view (trend + breakdown)

In [9]:
daily = dash_df.groupby("date", as_index=False)["sales"].sum()
prod = dash_df.groupby("product", as_index=False)["sales"].sum()
reg  = dash_df.groupby("region", as_index=False)["sales"].sum()

px.line(daily, x="date", y="sales", title="Dashboard: Daily Total Sales").show()
px.bar(prod, x="product", y="sales", title="Dashboard: Total Sales by Product").show()
px.bar(reg,  x="region",  y="sales", title="Dashboard: Total Sales by Region").show()


### 5.5.4 Add filters (optional, if ipywidgets is available)

In [ ]:
# If this cell errors, skip it — filtering still works with normal pandas.

try:
    import ipywidgets as widgets
    from IPython.display import display

    region_dd = widgets.Dropdown(
        options=["All"] + sorted(dash_df["region"].unique().tolist()),
        value="All", description="Region:"
    )
    product_dd = widgets.Dropdown(
        options=["All"] + sorted(dash_df["product"].unique().tolist()),
        value="All", description="Product:"
    )

    def update_plots(region, product):
        temp = dash_df.copy()
        if region != "All":
            temp = temp[temp["region"] == region]
        if product != "All":
            temp = temp[temp["product"] == product]

        daily = temp.groupby("date", as_index=False)["sales"].sum()
        prod  = temp.groupby("product", as_index=False)["sales"].sum()
        reg   = temp.groupby("region", as_index=False)["sales"].sum()

        px.line(daily, x="date", y="sales", title=f"Daily Sales (Region={region}, Product={product})").show()
        px.bar(prod, x="product", y="sales", title="Sales by Product").show()
        px.bar(reg,  x="region",  y="sales", title="Sales by Region").show()

    ui = widgets.HBox([region_dd, product_dd])
    out = widgets.interactive_output(update_plots, {"region": region_dd, "product": product_dd})
    display(ui, out)

except Exception as e:
    print("ipywidgets not available or not enabled. Error:", e)
    print("No problem — continue using normal pandas filtering.")


Output()

---  
## 5.6 Storytelling with Data (Make the message obvious)

### Storytelling tools
- **Headline titles**: say the conclusion (not just “Sales Chart”)
- **Annotations**: point to key events
- **Highlight**: focus on one thing
- **Reference lines**: compare against average/target


### 5.6.1 Headline title + annotation (Plotly)

In [11]:
# Create a storyline: add a promotion spike to the daily series
story = daily.copy()
promo_day = story["date"].iloc[45]
story.loc[story["date"] == promo_day, "sales"] *= 1.4

fig = px.line(story, x="date", y="sales",
              title="Promotion increased sales noticeably around mid-period")
fig.add_annotation(
    x=promo_day,
    y=float(story.loc[story["date"] == promo_day, "sales"].iloc[0]),
    text="Promo day",
    showarrow=True,
    arrowhead=2
)
fig.show()


### 5.6.2 Highlight one category (focus attention)

In [12]:
temp = dash_df.groupby(["date","product"], as_index=False)["sales"].sum()

fig = px.line(temp, x="date", y="sales", color="product",
              title="Compare product trends (focus on one product in discussion)")
fig.show()


### 5.6.3 Add a reference line (average)

In [13]:
avg_line = story["sales"].mean()

fig = px.line(story, x="date", y="sales", title="Sales with reference line (average)")
fig.add_hline(y=avg_line, annotation_text="Average", annotation_position="top left")
fig.show()


---  
## Mini Project (15–20 min): Build a 'Dashboard + Story'

### Task
1) Pick one **region** and one **product**  
2) Create:
   - A KPI: total sales
   - A trend line
   - A breakdown bar chart (either product or region)
3) Write **2 sentences** explaining the story from your charts


### Starter code (students fill region/product)

In [ ]:
region_choice = "East"
product_choice = "C"

temp = dash_df[(dash_df["region"] == region_choice) & (dash_df["product"] == product_choice)]

kpi_total = temp["sales"].sum()
daily_temp = temp.groupby("date", as_index=False)["sales"].sum()

print("KPI - Total sales:", round(kpi_total, 2))

px.line(daily_temp, x="date", y="sales",
        title=f"{region_choice}-{product_choice}: Daily Sales Trend").show()


### Practice (no solutions shown)
1) Create a bar chart of total sales by **product** for only the **West** region.  
2) Add an annotation to the day with the **maximum** sales.  
3) Make a headline title that states a conclusion (example: “Sales peaked after promotion”).
